# Import libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, 
                                    Input, 
                                    LSTM, 
                                    Dropout, 
                                    Conv1D, 
                                    MaxPooling1D, 
                                    Flatten
                                )
from tensorflow.keras.optimizers import (Adam, 
                                         AdamW)
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report

/Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/tensorflow_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
print("TensorFlow version:", tf.__version__)
print("Available physical devices:")
print(tf.config.list_physical_devices())

print("\nIs MPS available?")
print(tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
Available physical devices:
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Is MPS available?
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
data = pd.read_pickle("../data/processed/eeg_bandpass_ica.pkl")
data = data[data["shape"]==(64, 656)]

#### Step 1 : Setting Data in Appropriate Shape

In [4]:
X = np.stack(data["epoch_data_ICA"].values, axis = 0)
# Transpose to match (num_trials, n_samples, n_channels)
X = np.transpose(X, (0, 2, 1))

y = data["label"].values

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

Shape of X: (4083, 656, 64)
Shape of y: (4083,)


In [5]:
X

array([[[ 0.00000000e+00,  5.71069103e-16, -8.15813004e-17, ...,
          2.58340785e-16, -6.39053520e-16,  5.16681569e-16],
        [ 2.10483546e-01,  1.19956916e-01,  8.67188225e-02, ...,
          4.58757253e-01,  4.86667284e-01,  3.43769899e-01],
        [ 2.98658312e-01,  2.18957884e-01,  1.78514966e-01, ...,
          9.41222708e-01,  9.75980129e-01,  7.79775064e-01],
        ...,
        [-5.69310800e-01, -6.33127493e-01, -6.56823575e-01, ...,
         -1.32173184e+00, -1.18088383e+00, -1.59731625e+00],
        [-2.99838445e-01, -3.88125443e-01, -3.64295266e-01, ...,
         -1.16747458e+00, -1.04187114e+00, -1.55176020e+00],
        [-4.89487802e-16, -3.53518968e-16, -3.80712735e-16, ...,
         -3.67115852e-16,  3.67115852e-16,  3.80712735e-16]],

       [[ 4.32658682e-16, -3.89081549e-17, -6.22530478e-18, ...,
         -4.73123163e-16, -8.71542669e-16,  4.98024382e-17],
        [-4.45334059e-02, -8.01186418e-02, -1.81517106e-02, ...,
         -1.53590901e-01, -3.17212529e

#### Step -2 : Normalize Data

In [6]:
# Global Normalization
X = (X-np.mean(X))/np.std(X)

In [7]:
X.shape[0]

4083

#### Step 3: One-hot Encode labels (for classification)

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y)

# Transform 
y_le = le.transform(y)

# Applying to_categorical 
y_le = to_categorical(y_le, num_classes= len(np.unique(y_le)))
print(f"Shape of y : {y_le.shape}")

Shape of y : (4083, 2)


#### Step 4. Creating tensorflow Dataset

In [9]:
data = tf.data.Dataset.from_tensor_slices((X, y_le))

2025-10-06 22:12:51.950958: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2025-10-06 22:12:51.969863: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2025-10-06 22:12:51.970471: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
I0000 00:00:1759806771.972592 2698752 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1759806771.973571 2698752 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


#### Step 5: Splitting the Tensorflow Dataset into train, test and valid

In [10]:
train_split_ratio = 0.7
val_split_ratio = 0.15
test_split_ratio = 0.15

# Size of each split
train_size = int(train_split_ratio * X.shape[0])
val_size = int(val_split_ratio * X.shape[0])
test_size = int(test_split_ratio * X.shape[0])

# Shuffle the dataset first for a random split
data = data.shuffle(buffer_size= X.shape[0])

training_dataset = data.take(train_size)
val_dataset = data.take(val_size)
test_dataset = data.take(test_size)

print(f"Train dataset size: {len(list(training_dataset.as_numpy_iterator()))}")
print(f"Val dataset size: {len(list(val_dataset.as_numpy_iterator()))}")
print(f"Test dataset size: {len(list(test_dataset.as_numpy_iterator()))}")

Train dataset size: 2858
Val dataset size: 612
Test dataset size: 612


2025-10-06 22:12:56.220571: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-10-06 22:12:56.278172: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
training_dataset = training_dataset.batch(32)
val_dataset = val_dataset.batch(32)
test_dataset = test_dataset.batch(32)

In [12]:
for index, epoch in enumerate(training_dataset.take(1)):
    print(epoch[0].shape)

(32, 656, 64)


2025-10-06 22:12:59.722548: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


#### Step 6 : CNN MODEL ARCHITECTURE - USING MODEL SUBCLASSING

In [13]:
class LSTMModel(Model):

    def __init__(self, num_classes):
        super(LSTMModel, self).__init__()

        # LSTM and DropOut layer with 80 hidden units
        self.lstm = LSTM(units=100, 
                         activation = "tanh",
                         return_sequences=False)
        self.dropout = Dropout(rate=0.2)

        # Fully connected layer (2 units for binary classification)
        self.fc_1 = Dense(units=num_classes, activation='leaky_relu')
        self.fc_2 = Dense(units=num_classes, activation='leaky_relu')
        self.out = Dense(units=num_classes, activation='sigmoid')  # softmax for multi-class

    def call(self, inputs, training=False):
        x = self.lstm(inputs)
        x = self.dropout(x, training=training)
        
        # Dense layer 1
        x = self.fc_1(x)
        x = self.fc_2(x)

        # Output layer
        out = self.out(x)
        return out

# Instantiate model
num_classes = 2  # Change if you have more classes
lstm_model = LSTMModel(num_classes=num_classes)

In [14]:
print(lstm_model.summary())

f1_metric = tf.keras.metrics.F1Score(average='macro')

# Compile the Model
lstm_model.compile(optimizer= Adam(), 
                        loss = "categorical_crossentropy", 
                        metrics = ["accuracy", f1_metric])

Model: "lstm_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [15]:
history = lstm_model.fit(
    training_dataset,
    epochs = 40, 
    batch_size = 32, 
    validation_data = val_dataset
)

Epoch 1/40


2025-10-06 22:13:11.741762: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


90/90 ━━━━━━━━━━━━━━━━━━━━ 10s 63ms/step - accuracy: 0.4874 - f1_score: 0.4782 - loss: 0.6956 - val_accuracy: 0.5147 - val_f1_score: 0.4560 - val_loss: 0.6923
Epoch 2/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.5266 - f1_score: 0.5208 - loss: 0.6927 - val_accuracy: 0.5441 - val_f1_score: 0.5004 - val_loss: 0.6909
Epoch 3/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.5180 - f1_score: 0.5078 - loss: 0.6908 - val_accuracy: 0.5392 - val_f1_score: 0.4451 - val_loss: 0.6920
Epoch 4/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.5142 - f1_score: 0.4705 - loss: 0.6922 - val_accuracy: 0.5474 - val_f1_score: 0.5451 - val_loss: 0.6874
Epoch 5/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 47ms/step - accuracy: 0.5372 - f1_score: 0.5349 - loss: 0.6910 - val_accuracy: 0.5408 - val_f1_score: 0.5335 - val_loss: 0.6862
Epoch 6/40
90/90 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - accuracy: 0.5580 - f1_score: 0.5246 - loss: 0.6889 - val_accuracy: 0.5474 - val_f1_score: 0.5277 - val_loss: 0.6

In [16]:
# Test Predictions
y_val_pred = lstm_model.predict(val_dataset)
y_val_pred = np.argmax(y_val_pred, axis = 1)

# Original Predictions
y_val_original = np.concatenate([y.numpy() for x, y in val_dataset])
y_val_original = np.argmax(y_val_original, axis = 1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step


In [17]:
print(classification_report(y_val_original, y_val_pred))

              precision    recall  f1-score   support

           0       0.49      0.49      0.49       310
           1       0.48      0.48      0.48       302

    accuracy                           0.49       612
   macro avg       0.49      0.49      0.49       612
weighted avg       0.49      0.49      0.49       612



# Test Predictions

In [18]:
# Test Predictions
y_test_pred = lstm_model.predict(test_dataset)
y_test_pred = np.argmax(y_test_pred, axis = 1)

# Original Predictions
y_test_original = np.concatenate([y.numpy() for x, y in test_dataset])
y_test_original = np.argmax(y_test_original, axis = 1)

20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


In [19]:
print(classification_report(y_test_original, y_test_pred))

              precision    recall  f1-score   support

           0       0.47      0.44      0.45       310
           1       0.46      0.49      0.48       302

    accuracy                           0.47       612
   macro avg       0.47      0.47      0.47       612
weighted avg       0.47      0.47      0.47       612

